In [1]:
import tensorflow as tf
import numpy as np
import tensorflow_recommenders as tfrs
import tensorflow_ranking as tfr


In [2]:

def global_average_mean(x):
  """Custom layer to perform global average mean pooling."""
  axis = -2  # Reduce mean along the last dimension
  return tf.reduce_mean(x, axis=axis)


In [3]:

def reshaper(x):
    shape = (-1,10,1)  
    return tf.reshape(x, shape)


In [4]:

class ItemModel(tf.keras.Model):
    def __init__(
        self,
        unique_item_ids,
        unique_item_gics,
        unique_item_names

        ):
        super().__init__()

        self.max_tokens = 10000
        self.unique_item_ids = unique_item_ids
        self.unique_item_gics = unique_item_gics
        self.unique_item_names = unique_item_names

        self.embed_item_id = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary = self.unique_item_ids,
                mask_token =None
            ),
            tf.keras.layers.Embedding(
                input_dim = len(self.unique_item_ids)+1,
                output_dim = 16 #32
            )
        ])

        self.embed_items_gics = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary = unique_item_gics,
                mask_token = None
            ),
            tf.keras.layers.Embedding(
                input_dim = len(unique_item_gics)+1,
                output_dim = 16 #len(unique_item_gics)
            )
        ])

        self.textvectorizer = tf.keras.layers.TextVectorization(
            max_tokens = self.max_tokens,
            # ragged = True
        )

        self.embed_item_name = tf.keras.Sequential([

            # tf.keras.layers.Reshape((-1,5,1)),
            tf.keras.layers.Lambda(reshaper),

            self.textvectorizer,

            tf.keras.layers.Embedding(
                input_dim = self.max_tokens,
                output_dim = 32,
                mask_zero = True
            ),

            tf.keras.layers.Lambda(global_average_mean)
            # tf.keras.layers.GlobalAveragePooling1D(), # reduces dimensionality to 1d (embedding layer embeddeds each word in a title one by one)
            
            # tf.keras.layers.Flatten() 
            # squeeze_custom_layer()
        ])

        self.textvectorizer.adapt(self.unique_item_names)

    def call(self, inputs):

        item_id,  item_gics, item_name = inputs

        return tf.concat([
            self.embed_item_id(item_id),
            self.embed_items_gics(item_gics),
            self.embed_item_name(item_name)
        ],
        axis = 2)



In [5]:
class UserModel(tf.keras.Model):
    def __init__(
        self,
        unique_item_ids):

        super().__init__()

        self.unique_item_ids = unique_item_ids
        
        self.embed_user_id = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary = self.unique_item_ids,
                mask_token = None
            ),
            tf.keras.layers.Embedding(
                input_dim = len(self.unique_item_ids)+1,
                output_dim = 32
            ),
            tf.keras.layers.GlobalAveragePooling1D()
    
        ])
        
    def call(self, inputs):

        user_id = inputs["USER_ID"]

        return self.embed_user_id(user_id)

In [6]:
def generate_embedding(item, item_model):
  
      item_id = item['STOCKCODE'] 
      item_name = item['STOCKNAME']
      item_gics = item['GICS']

      embedding = item_model([item_id, item_name, item_gics])
      return embedding

In [7]:

class Ranker(tfrs.Model):

  def __init__(self, portfolios, loss):
    super().__init__()

    embedding_dimension = 32,
    self.loss = loss
    
    self.portfolios = portfolios

    self.items_ids = self.portfolios.batch(10000).map(lambda x: x["STOCKCODE"])
    self.unique_item_ids = np.unique(np.concatenate(list(self.items_ids)))

    self.item_GICS = self.portfolios.batch(10000).map(lambda x: x["GICS"])
    self.unique_item_gics = np.unique(np.concatenate(list(self.item_GICS)))

    self.item_names = portfolios.batch(10000).map(lambda x: x["STOCKNAME"])
    self.unique_item_names = np.unique(np.concatenate(list(self.item_names)))
    
    # self.user_ids = self.portfolios.batch(10000).map(lambda x: x["CDSACCNO"])
    # self.unique_user_ids = np.unique(np.concatenate(list(self.user_ids)))

    self.user_ids = self.portfolios.batch(10000).map(lambda x: x["USER_ID"])
    self.unique_user_ids = np.unique(np.concatenate(list(self.user_ids)))


    self.user_embeddings = UserModel(
        unique_item_ids = self.unique_item_ids
    )

    self.item_embeddings = ItemModel(
        unique_item_ids = self.unique_item_ids,
        unique_item_gics = self.unique_item_gics,
        unique_item_names = self.unique_item_names
    )

    self.score_model = tf.keras.Sequential([
      tf.keras.layers.Dense(256, activation="relu"),
      tf.keras.layers.Dense(64, activation="relu"),
      tf.keras.layers.Dense(1)
    ])

    self.task = tfrs.tasks.Ranking(
      loss=loss,
      metrics=[
        tfr.keras.metrics.NDCGMetric(name="ndcg_metric"),
        tf.keras.metrics.RootMeanSquaredError()
      ]
    )

  def call(self, features):

    user_embeddings = self.user_embeddings(features["USER_ID"])

    item_embeddings = self.item_embeddings((features["STOCKCODE"], features['GICS'], features['STOCKNAME'])) #, features['STOCKNAME']

    list_length = features["STOCKCODE"].shape[1]
    user_embedding_repeated = tf.repeat(
        tf.expand_dims(user_embeddings, 1), [list_length], axis=1)

    # print(user_embedding_repeated.shape,' | ' ,item_embeddings.shape)
    
    concatenated_embeddings = tf.concat(
        [user_embedding_repeated, item_embeddings], 2)

    return self.score_model(concatenated_embeddings)

  def compute_loss(self, features, training=False):
    labels = features.pop("RATING")

    scores = self(features)

    return self.task(
        labels=labels,
        predictions=tf.squeeze(scores, axis=-1),
    )


# Training

In [8]:
import os
import numpy as np
import tensorflow as tf
import tensorflow_recommenders as tfrs
from ranker_recommender import Ranker
from datetime import datetime

base_loc = r'D:\dev work\recommender systems\ATRAD_CARS'

portfolios = tf.data.Dataset.load(r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_50_useridseq\portfolios").cache()
train_list_ds = tf.data.Dataset.load(r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_50_useridseq\ranker_train").cache()
test_list_ds = tf.data.Dataset.load(r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_50_useridseq\ranker_test").cache()

items_ids = portfolios.batch(10000).map(lambda x: x["STOCKCODE"])
item_names = portfolios.batch(10000).map(lambda x: x["STOCKNAME"])
item_GICS = portfolios.batch(10000).map(lambda x: x["GICS"])

user_ids = portfolios.batch(10000).map(lambda x: x["CDSACCNO"])

unique_item_ids = np.unique(np.concatenate(list(items_ids)))
unique_item_names = np.unique(np.concatenate(list(item_names)))
unique_item_gics = np.unique(np.concatenate(list(item_GICS)))

unique_user_ids = np.unique(np.concatenate(list(user_ids)))

# need these to initialize timestamp embedding layers in future steps

train_list_ds = train_list_ds.shuffle(10000).batch(256)
test_list_ds = test_list_ds.shuffle(10000).batch(128)


In [9]:
print (len(np.unique(np.concatenate(list(test_list_ds.map(lambda x: x['CDSACCNO']).as_numpy_iterator())))))
print (len(np.unique(np.concatenate(list(train_list_ds.map(lambda x: x['CDSACCNO']).as_numpy_iterator())))))

2984
2984


In [10]:
model = Ranker(
    # use_timestamp = True,
    loss = tf.keras.losses.MeanSquaredError(),
    portfolios = portfolios
    )

In [17]:
log_dir = os.path.join(base_loc ,"logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S"))
# log_dir = "../../logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=0,
    embeddings_freq = 1,
    write_images = True)

model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))

model.fit(
    train_list_ds, 
    epochs=5, 
    verbose = 1,
    callbacks=[tensorboard_callback]
    )

#save model
base = r'D:\dev work\recommender systems\ATRAD_CARS\model_weights\{}'.format(datetime.now().strftime("%Y_%m_%d"))

if not os.path.exists(base):
    os.makedirs(base)

model_name = 'tf_listwise_ranking_useridseq_{}'.format(datetime.now().strftime("%Y_%m_%d_%H_%M"))
save_path = os.path.join(base,model_name)

model.save_weights(save_path)

print()
print("saved model @ : {}".format(save_path))

Epoch 1/5
583/583 [==============================] - 8s 12ms/step - ndcg_metric: 0.8262 - root_mean_squared_error: 0.9196 - loss: 0.8455 - regularization_loss: 0.0000e+00 - total_loss: 0.8455
Epoch 2/5
583/583 [==============================] - 7s 12ms/step - ndcg_metric: 0.8302 - root_mean_squared_error: 0.8646 - loss: 0.7476 - regularization_loss: 0.0000e+00 - total_loss: 0.7476
Epoch 3/5
583/583 [==============================] - 7s 12ms/step - ndcg_metric: 0.8405 - root_mean_squared_error: 0.8407 - loss: 0.7068 - regularization_loss: 0.0000e+00 - total_loss: 0.7068
Epoch 4/5
583/583 [==============================] - 7s 12ms/step - ndcg_metric: 0.8714 - root_mean_squared_error: 0.7917 - loss: 0.6267 - regularization_loss: 0.0000e+00 - total_loss: 0.6267
Epoch 5/5
583/583 [==============================] - 7s 12ms/step - ndcg_metric: 0.9085 - root_mean_squared_error: 0.7170 - loss: 0.5139 - regularization_loss: 0.0000e+00 - total_loss: 0.5139

saved model @ : D:\dev work\recommender

In [14]:
print()
print("*** TESTING ***")
# test_list_ds = test_list_ds.batch(128)
model = model.evaluate(test_list_ds, return_dict=True)
print("NDCG of the MSE Model: {:.4f}".format(model["ndcg_metric"]))


*** TESTING ***
24/24 [==============================] - 1s 7ms/step - ndcg_metric: 0.7741 - root_mean_squared_error: 1.4394 - loss: 2.0769 - regularization_loss: 0.0000e+00 - total_loss: 2.0769
NDCG of the MSE Model: 0.7741


In [21]:
test_user = next(iter(test_list_ds.unbatch().batch(1)))
test_user

{'UNIX_TS': <tf.Tensor: shape=(1, 10), dtype=float32, numpy=
 array([[1.6655130e+09, 1.6659450e+09, 1.6650810e+09, 1.6680186e+09,
         1.6662042e+09, 1.6654266e+09, 1.6654266e+09, 1.6804602e+09,
         1.6801146e+09, 1.6649946e+09]], dtype=float32)>,
 'RATING': <tf.Tensor: shape=(1, 10), dtype=float32, numpy=array([[2., 4., 3., 4., 3., 2., 2., 4., 3., 5.]], dtype=float32)>,
 'STOCKCODE': <tf.Tensor: shape=(1, 10), dtype=string, numpy=
 array([[b'REEF', b'DIPD', b'BIL', b'MGT', b'HAYC', b'EXPO', b'PACK',
         b'ASPH', b'SEMB', b'RCL']], dtype=object)>,
 'USER_ID': <tf.Tensor: shape=(1, 10), dtype=string, numpy=
 array([[b'RCH', b'HSIG', b'GHLL', b'AINS', b'AMSL', b'RCL', b'BIL',
         b'PACK', b'EXPO', b'REEF']], dtype=object)>,
 'GICS': <tf.Tensor: shape=(1, 10), dtype=string, numpy=
 array([[b'Consumer Services', b'Materials', b'Food Beverage & Tobacco',
         b'Consumer Durables & Apparel', b'Materials', b'Transpotation',
         b'Materials', b'Materials', b'Diversi

In [25]:
pred_ratings = model(test_user)
pred_ratings

<tf.Tensor: shape=(1, 10, 1), dtype=float32, numpy=
array([[[1.4253658],
        [2.9694061],
        [5.96837  ],
        [4.143403 ],
        [3.2110763],
        [6.757991 ],
        [1.4270476],
        [1.6193805],
        [3.5014644],
        [2.892148 ]]], dtype=float32)>

In [ ]:
recommendations_w_ratings = pd.DataFrame()
recommendations_w_ratings['STOCKCODE'] = recommendations
recommendations_w_ratings['PRED_RATING'] = pred_ratings.numpy().flatten()
recommendations_w_ratings = recommendations_w_ratings.sort_values( by = ['PRED_RATING'], ascending= False).reset_index(drop = True)

user_test_port_ = test_df.iloc[test_users_.groups[CDSACCNO]].sort_values('RATING', ascending = False)
recommendations_w_ratings = recommendations_w_ratings.join(user_test_port_.set_index('STOCKCODE'), on = 'STOCKCODE')[['STOCKCODE','PRED_RATING','RATING']]

hit_perc = (10 - recommendations_w_ratings['RATING'].isna().sum())/len(recommendations_w_ratings)

return recommendations_w_ratings, hit_perc
